# Train Nik 0.1: a tiny romanized-Nepali LLM, from scratch

Runs on a free **Colab T4** or **Kaggle T4/P100** GPU. Turn on the GPU first (Colab: Runtime → Change runtime type → T4; Kaggle: Settings → Accelerator → GPU, Internet on).

The pipeline downloads the data, builds the romanized corpus, trains the tokenizer, pretrains, runs SFT, exports int8 weights under 10 MB and updates `nik/web/model/manifest.json`, so `/chat` picks up the new model as soon as you push.

In [ ]:
!nvidia-smi -L || echo 'no GPU: training will be slow'
!git clone https://github.com/NikitHamal/NikitHamal.git
%cd NikitHamal
!pip -q install pyarrow numpy

## Settings
`PRESET`: `nik-0.1` (~8.3M params, the shipped model), `nik-mini` (~4M) or `nik-nano` (~2M).
On a T4, 30k pretrain steps plus 6k SFT steps at batch 48 takes about 2.5 h. Halve the steps for a quick run.

In [ ]:
import os
os.environ['PRESET'] = 'nik-0.1'
os.environ['SHARDS'] = '2'          # news shards (~190M Devanagari chars each)
os.environ['PRE_STEPS'] = '30000'
os.environ['SFT_STEPS'] = '6000'
os.environ['BS'] = '48'
os.environ['COMPILE'] = '1'         # torch.compile (unset if it errors)

In [ ]:
!bash nik/scripts/pipeline.sh 2>&1 | tee nik_train.log

## Try it in Python

In [ ]:
import sys, torch
sys.path.insert(0, 'nik')
from export import load_exported
from tokenizer import Tokenizer
from prepare import render_chat
from model import generate_ssbd
m = load_exported(f"nik/web/model/{os.environ['PRESET']}.bin")
tok = Tokenizer.load('nik/web/model/tokenizer.json')
for q in ['timi ko hau?', '47 + 38 kati hunchha?', 'nikit hamal ko ho?', 'namaste! k chha?']:
    ids, _ = render_chat(tok, [{'role': 'user', 'content': q}])
    out, passes = generate_ssbd(m, ids[:-1] + [6], 100)
    print('Q:', q, '\nA:', tok.decode(out), f'[{len(out)} tok / {passes} passes]\n')

## Push the new model to GitHub
Create a fine-grained token with *Contents: read & write* on this repo, then run the cell below. Alternatively, download `nik/web/model/` and commit it from your machine.

In [ ]:
from getpass import getpass
token = getpass('GitHub token: ')
!git config user.name 'Nikit Hamal' && git config user.email 'iamnikithamal@gmail.com'
!git add nik/web/model nik_train.log -f
!git commit -m "nik: trained $PRESET on GPU" || true
!git push https://{token}@github.com/NikitHamal/NikitHamal.git HEAD:main